# 카카오맵 정류소·역 좌표 수집 → DuckDB 적재

수도권(서울·인천·경기) 정류소 + 지하철역 좌표를 카카오맵 내부 검색 API로 일괄 수집.

## API
- 엔드포인트: `https://search.map.kakao.com/mapsearch/map.daum`
- 응답: JSONP (`callback(...)` wrapper) → 정규식으로 JSON 추출
- **쿠키 불필요** (검증됨)
- 키 인덱스: `busStop` 배열 (정류소 전용), `place` 배열 (POI), 등

## 매칭 전략 (검증됨, 10/10 100%)
1. **1차**: `f"{sgg_nm} {sttn_nm}"` 검색 → busStop에서 `RGN_D2_NAME == sgg_nm` 찾기
2. **2차** (1차 0건): `f"{sttn_nm}"` 검색 → busStop에서 `RGN_D2_NAME == sgg_nm` 찾기
3. **미매칭**: `matched_kind='none'` 마킹 (재시도 가능)

## 캐시 키 = `(ctpv_cd, sgg_cd, sttn_nm)`
- 같은 시군 같은 이름 → 같은 정류소 (양방향 ±10m로 가정)
- 다른 시군 같은 이름 → 별개 정류소 (disambiguation 필요)

## 안전장치
- ThreadPoolExecutor 동시 호출 (사용자 의견: 30) + sleep 2s+jitter
- 배치마다 DB INSERT (체크포인트)
- 재개 가능 (이미 적재된 키 자동 skip)
- 차단 감지 (HTTP 5xx, JSONP 파싱 실패 누적 시 중단)

## 예상 수치
- 대상 unique 키: ~35,000~45,000
- 호출 시간: ~1~2시간

In [1]:
# ============================================================
# 셀 1 - 환경 + DB 헬퍼
# ============================================================
import os
import re
import json
import time
import random
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"DB: {DB_PATH}")

DB: data/seoul.duckdb


In [8]:
# ============================================================
# 셀 2 - 카카오맵 검색 + JSONP 파서 + 매칭 함수
# ============================================================
# 매칭 전략 (검증됨):
#   1차: f"{ctpv_short} {sgg_nm} {sttn_nm}" → busStop에서 RGN_D1+RGN_D2 일치하는 것
#   2차: f"{sgg_nm} {sttn_nm}"              → busStop에서 RGN_D1+RGN_D2 일치하는 것
#   미매칭: 'none'
# 시도 prefix 필수: 동명 sgg(서울 중구 vs 부산 중구, 인천 동구 vs 부산 동구) 구별

KAKAO_URL = "https://search.map.kakao.com/mapsearch/map.daum"
HEADERS = {
    "accept": "*/*",
    "accept-language": "ko-KR,ko;q=0.9",
    "referer": "https://map.kakao.com/",
    "user-agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/147.0.0.0 Safari/537.36"
    ),
}

# 시도코드 → 카카오 RGN_D1_NAME에서 쓰이는 짧은 시도명
CTPV_SHORT = {
    "11": "서울", "26": "부산", "27": "대구", "28": "인천",
    "29": "광주", "30": "대전", "31": "울산", "36": "세종",
    "41": "경기", "42": "강원", "43": "충북", "44": "충남",
    "45": "전북", "46": "전남", "47": "경북", "48": "경남", "50": "제주",
}

_JSONP_RE = re.compile(r'^[^(]+\((.*)\)\s*;?\s*$', re.DOTALL)


def parse_jsonp(text: str) -> Optional[dict]:
    m = _JSONP_RE.match(text)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None


def kakao_search(query: str, timeout: int = 10) -> Optional[dict]:
    try:
        r = requests.get(
            KAKAO_URL, headers=HEADERS,
            params={"callback": "jQuery1", "q": query, "msFlag": "A", "sort": "0"},
            timeout=timeout,
        )
        if r.status_code != 200:
            return None
        return parse_jsonp(r.text)
    except (requests.Timeout, requests.ConnectionError):
        return None


def _pick_busStop(j: dict, ctpv_short: str, sgg_nm: str) -> Optional[dict]:
    """busStop 배열에서 시도+시군 모두 일치하는 첫 결과."""
    bs = j.get("busStop") or []
    return next(
        (b for b in bs
         if b.get("RGN_D1_NAME", "").startswith(ctpv_short)
         and b.get("RGN_D2_NAME") == sgg_nm),
        None,
    )


def _pack(matched: dict, kind: str, query_used: str) -> dict:
    return {
        "matched_kind": kind,
        "matched_name": matched.get("BUSSTOP_NAME"),
        "lat": float(matched["lat"]) if matched.get("lat") else None,
        "lon": float(matched["lon"]) if matched.get("lon") else None,
        "rgn_d1": matched.get("RGN_D1_NAME"),
        "rgn_d2": matched.get("RGN_D2_NAME"),
        "query_used": query_used,
    }


def find_stop_coords(ctpv_cd: str, sgg_nm: str, sttn_nm: str) -> dict:
    """버스 정류소 좌표 매칭 (시도 prefix + RGN_D1/D2 검증)."""
    ctpv_short = CTPV_SHORT.get(ctpv_cd)
    if ctpv_short is None:
        return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
                "rgn_d1": None, "rgn_d2": None, "query_used": ""}

    # 1차: 시도 + 시군 + 정류소명
    q1 = f"{ctpv_short} {sgg_nm} {sttn_nm}"
    j = kakao_search(q1)
    if j is None:
        return {"matched_kind": "error", "matched_name": None, "lat": None, "lon": None,
                "rgn_d1": None, "rgn_d2": None, "query_used": q1}
    m = _pick_busStop(j, ctpv_short, sgg_nm)
    if m:
        return _pack(m, "busStop_with_sgg", q1)

    # 2차: 시군 + 정류소명 (시도 단어 없이 다시) — 필터는 유지
    q2 = f"{sgg_nm} {sttn_nm}"
    j = kakao_search(q2)
    if j is None:
        return {"matched_kind": "error", "matched_name": None, "lat": None, "lon": None,
                "rgn_d1": None, "rgn_d2": None, "query_used": q2}
    m = _pick_busStop(j, ctpv_short, sgg_nm)
    if m:
        return _pack(m, "busStop_fallback", q2)

    return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
            "rgn_d1": None, "rgn_d2": None, "query_used": q1}


def find_subway_coords(ctpv_cd: str, sttn_nm: str, rte_nm: Optional[str] = None) -> dict:
    """지하철역 좌표 매칭. rte_nm 있으면 같이 검색해 정확도 향상.
    
    수도권은 ctpv_cd로 시도 필터 (서울/인천/경기). RGN_D2_NAME은 검증 안 함
    (지하철역의 행정구역은 정류소와 다를 수 있어서 시도만 일치하면 OK).
    """
    ctpv_short = CTPV_SHORT.get(ctpv_cd)
    if ctpv_short is None:
        return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
                "rgn_d1": None, "rgn_d2": None, "query_used": ""}

    # 1차: 시도 + 정류소명 + 노선 (가장 정확)
    queries = [f"{ctpv_short} {sttn_nm} 지하철역"]
    if rte_nm:
        queries.insert(0, f"{ctpv_short} {sttn_nm} {rte_nm}")
    
    for q in queries:
        j = kakao_search(q)
        if j is None:
            continue
        # busStop 우선, 없으면 place 시도
        bs = j.get("busStop") or []
        m = next((b for b in bs if b.get("RGN_D1_NAME", "").startswith(ctpv_short)), None)
        if m:
            return _pack(m, "subway_strict", q)
        pl = j.get("place") or []
        m = next(
            (p for p in pl
             if p.get("name", "").startswith(sttn_nm)
             and ctpv_short in (p.get("address") or "")),
            None,
        )
        if m:
            return {
                "matched_kind": "subway_place",
                "matched_name": m.get("name"),
                "lat": float(m["lat"]) if m.get("lat") else None,
                "lon": float(m["lon"]) if m.get("lon") else None,
                "rgn_d1": ctpv_short,
                "rgn_d2": None,
                "query_used": q,
            }

    return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
            "rgn_d1": None, "rgn_d2": None, "query_used": queries[0]}


print("검색 함수 준비 완료 (find_stop_coords + find_subway_coords)")

검색 함수 준비 완료 (find_stop_coords + find_subway_coords)


In [3]:
# ============================================================
# 셀 3 - 테이블 생성 + 수도권 대상 추출 + 재개 목록
# ============================================================
# region 테이블 join → sgg_nm 정규화 ("서울특별시 종로구" → "종로구")
# 수도권: ctpv_cd IN ('11', '28', '41')

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS stop_coords (
        ctpv_cd        TEXT,
        sgg_cd         TEXT,
        sgg_nm         TEXT,
        sttn_nm        TEXT,
        matched_name   TEXT,
        lat            DOUBLE,
        lon            DOUBLE,
        rgn_d1         TEXT,
        rgn_d2         TEXT,
        matched_kind   TEXT,
        query_used     TEXT,
        fetched_at     TIMESTAMP,
        PRIMARY KEY (ctpv_cd, sgg_cd, sttn_nm)
    )
    """)

with db_open(read_only=True) as con:
    # 우리 DB의 unique (ctpv_cd, sgg_cd, sttn_nm) + region 테이블에서 sgg_nm 정규화
    targets = con.execute("""
        WITH norm_region AS (
            SELECT sido_cd AS ctpv_cd,
                   sido_cd || sgg_cd AS sgg_full,
                   -- 시도명 prefix 제거 ("서울특별시 ", "인천광역시 ", "경기도 " 등)
                   regexp_replace(locatadd_nm, '^(서울특별시|인천광역시|경기도|강원도|강원특별자치도|충청북도|충청남도|전라북도|전라남도|전북특별자치도|경상북도|경상남도|제주특별자치도|광주광역시|대전광역시|대구광역시|울산광역시|부산광역시|세종특별자치시) ', '') AS sgg_nm
            FROM region
            WHERE sgg_cd <> '000' AND umd_cd = '000' AND ri_cd = '00'
        )
        SELECT DISTINCT 
            rs.ctpv_cd,
            rs.sgg_cd,
            nr.sgg_nm,
            rs.sttn_nm
        FROM route_station rs
        LEFT JOIN norm_region nr
          ON nr.sgg_full = rs.ctpv_cd || substr(rs.sgg_cd, 3)  -- sgg_cd가 11110 형식
         OR nr.sgg_full = rs.sgg_cd                              -- 또는 5자리 그대로
        WHERE rs.ctpv_cd IN ('11', '28', '41')   -- 수도권: 서울/인천/경기
          AND rs.sttn_nm IS NOT NULL
          AND nr.sgg_nm IS NOT NULL
    """).df()
    
    done = con.execute("""
        SELECT ctpv_cd, sgg_cd, sttn_nm FROM stop_coords
    """).df()

if len(done) > 0:
    done_keys = set(zip(done['ctpv_cd'], done['sgg_cd'], done['sttn_nm']))
else:
    done_keys = set()

targets['key'] = list(zip(targets['ctpv_cd'], targets['sgg_cd'], targets['sttn_nm']))
remaining = targets[~targets['key'].isin(done_keys)].drop(columns=['key']).reset_index(drop=True)

print(f"수도권 unique (ctpv_cd, sgg_cd, sttn_nm): {len(targets):,}개")
print(f"  ㄴ ctpv_cd 분포:")
print(targets.groupby('ctpv_cd').size().to_string())
print(f"\n이미 적재: {len(done):,}개")
print(f"남은 작업: {len(remaining):,}개")
print(f"\n샘플 5개:")
print(remaining.head().to_string())

수도권 unique (ctpv_cd, sgg_cd, sttn_nm): 34,723개
  ㄴ ctpv_cd 분포:
ctpv_cd
11     8196
28     3755
41    22772

이미 적재: 0개
남은 작업: 34,723개

샘플 5개:
  ctpv_cd sgg_cd sgg_nm         sttn_nm
0      41  41550    안성시            수자원앞
1      41  41550    안성시              계동
2      41  41550    안성시           간목이입구
3      41  41550    안성시  한일타운.KCC스위첸아파트
4      41  41550    안성시          아이씨디입구


In [ ]:
# ============================================================
# 셀 4 - 본 수집: ThreadPool batch + sleep + 체크포인트 + 진행률
# ============================================================
# - 배치 단위(BATCH_SIZE개)로 동시 호출 → 모두 완료 후 DB INSERT → sleep
# - 차단 감지: 연속 ERROR 임계치 초과 시 즉시 중단
# - 재개: cell 3 다시 실행하면 done_keys 갱신되어 자동 skip

BATCH_SIZE = 30      # 동시 호출 수
SLEEP_BASE = 2.0     # 배치 후 기본 sleep
SLEEP_JITTER = 1.0   # 0~JITTER 랜덤 추가 (봇 패턴 회피)
MAX_CONSEC_ERRORS = 30
PROGRESS_EVERY = 5   # 5 batch마다 진행률 (= 150건)


def process_one(row) -> dict:
    res = find_stop_coords(row.ctpv_cd, row.sgg_nm, row.sttn_nm)
    return {
        "ctpv_cd": row.ctpv_cd,
        "sgg_cd": row.sgg_cd,
        "sgg_nm": row.sgg_nm,
        "sttn_nm": row.sttn_nm,
        **res,
        "fetched_at": datetime.now(),
    }


started_at = datetime.now()
stats = {"busStop_with_sgg": 0, "busStop_fallback": 0, "none": 0, "error": 0}
consec_errors = 0
stopped_early = False

n = len(remaining)
if n == 0:
    print("남은 작업 없음 — 모두 완료")
else:
    for batch_idx, start in enumerate(range(0, n, BATCH_SIZE), start=1):
        batch = remaining.iloc[start:start+BATCH_SIZE]
        results = []
        with ThreadPoolExecutor(max_workers=BATCH_SIZE) as ex:
            futures = [ex.submit(process_one, r) for r in batch.itertuples(index=False)]
            for f in as_completed(futures):
                results.append(f.result())

        batch_errors = 0
        for r in results:
            stats[r["matched_kind"]] = stats.get(r["matched_kind"], 0) + 1
            if r["matched_kind"] == "error":
                batch_errors += 1
        consec_errors = consec_errors + batch_errors if batch_errors == len(results) else 0

        if consec_errors >= MAX_CONSEC_ERRORS:
            print(f"\n🚨 연속 에러 {consec_errors}건 감지 — 차단 가능성. 중단.")
            stopped_early = True
            break

        df = pd.DataFrame(results)
        with db_open() as con:
            con.register("df_view", df)
            con.execute("""
                INSERT INTO stop_coords
                SELECT ctpv_cd, sgg_cd, sgg_nm, sttn_nm,
                       matched_name, lat, lon, rgn_d1, rgn_d2,
                       matched_kind, query_used, fetched_at
                FROM df_view
                ON CONFLICT (ctpv_cd, sgg_cd, sttn_nm) DO NOTHING
            """)
            con.unregister("df_view")

        done_now = min(start + BATCH_SIZE, n)
        if batch_idx % PROGRESS_EVERY == 0 or done_now == n:
            elapsed = datetime.now() - started_at
            pct = done_now / n * 100
            eta_sec = elapsed.total_seconds() / done_now * (n - done_now)
            total = sum(stats.values())
            match = (stats["busStop_with_sgg"] + stats["busStop_fallback"]) / max(total, 1) * 100
            print(f"[{done_now:>6,}/{n:,}] {pct:5.1f}% | 매칭 {match:.1f}% | "
                  f"1차 {stats['busStop_with_sgg']:>5,} / 2차 {stats['busStop_fallback']:>4,} / "
                  f"미매칭 {stats['none']:>4,} / 에러 {stats['error']:>3,} | "
                  f"경과 {str(elapsed).split('.')[0]} | ETA ~{int(eta_sec//60)}분")

        time.sleep(SLEEP_BASE + random.random() * SLEEP_JITTER)

elapsed = datetime.now() - started_at
print(f"\n{'중단됨' if stopped_early else '완료'} | 소요: {elapsed}")
print(f"최종 통계: {stats}")

with db_open(read_only=True) as con:
    grand = con.execute("SELECT COUNT(*) FROM stop_coords").fetchone()[0]
print(f"\n📊 stop_coords 총 누적: {grand:,}개")

In [ ]:
# ============================================================
# 셀 5 - error 마킹된 행 재시도
# ============================================================
# 'none' (진짜 검색 결과 없음)은 재시도 무의미 → 'error'만 다시

with db_open(read_only=True) as con:
    err_df = con.execute("""
        SELECT ctpv_cd, sgg_cd, sgg_nm, sttn_nm
        FROM stop_coords
        WHERE matched_kind = 'error'
    """).df()

if len(err_df) == 0:
    print("error 행 없음 — 재시도 불필요")
else:
    print(f"재시도 대상: {len(err_df):,}개")
    retry_results = []
    for r in err_df.itertuples(index=False):
        res = find_stop_coords(r.ctpv_cd, r.sgg_nm, r.sttn_nm)
        retry_results.append({
            "ctpv_cd": r.ctpv_cd, "sgg_cd": r.sgg_cd, "sgg_nm": r.sgg_nm, "sttn_nm": r.sttn_nm,
            **res, "fetched_at": datetime.now(),
        })
        time.sleep(0.3 + random.random() * 0.5)
    
    df = pd.DataFrame(retry_results)
    with db_open() as con:
        con.register("df_view", df)
        con.execute("""
            DELETE FROM stop_coords
            WHERE (ctpv_cd, sgg_cd, sttn_nm) IN (
                SELECT ctpv_cd, sgg_cd, sttn_nm FROM df_view
            )
        """)
        con.execute("""
            INSERT INTO stop_coords
            SELECT ctpv_cd, sgg_cd, sgg_nm, sttn_nm,
                   matched_name, lat, lon, rgn_d1, rgn_d2,
                   matched_kind, query_used, fetched_at
            FROM df_view
        """)
        con.unregister("df_view")
    
    new_stats = pd.Series([r["matched_kind"] for r in retry_results]).value_counts()
    print(f"\n재시도 결과:")
    print(new_stats.to_string())

In [6]:
# ============================================================
# 셀 6 - 검증 + 분석 쿼리
# ============================================================
with db_open(read_only=True) as con:
    print("=== 매칭 종류별 분포 ===")
    print(con.execute("""
        SELECT matched_kind, COUNT(*) AS cnt,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
        FROM stop_coords
        GROUP BY matched_kind
        ORDER BY cnt DESC
    """).df())

    print("\n=== ctpv_cd별 매칭률 ===")
    print(con.execute("""
        SELECT ctpv_cd,
               COUNT(*) AS 전체,
               SUM(CASE WHEN matched_kind LIKE 'busStop%' THEN 1 ELSE 0 END) AS 매칭,
               ROUND(100.0 * SUM(CASE WHEN matched_kind LIKE 'busStop%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct
        FROM stop_coords
        GROUP BY ctpv_cd
        ORDER BY ctpv_cd
    """).df())

    print("\n=== 미매칭 정류소 샘플 (10개) ===")
    print(con.execute("""
        SELECT ctpv_cd, sgg_nm, sttn_nm, matched_kind
        FROM stop_coords
        WHERE matched_kind IN ('none', 'error')
        LIMIT 10
    """).df())

    print("\n=== 좌표 범위 sanity check (수도권) ===")
    print(con.execute("""
        SELECT 
            MIN(lat) AS lat_min, MAX(lat) AS lat_max,
            MIN(lon) AS lon_min, MAX(lon) AS lon_max
        FROM stop_coords
        WHERE lat IS NOT NULL
    """).df())
    print("  ※ 정상이면 lat 36.5~38.5, lon 126.5~127.7 범위")

    print("\n=== card_trip 정류소 매칭 적용 사전 검증 ===")
    print(con.execute("""
        WITH stops AS (
            SELECT DISTINCT ride_sttn_id AS sttn_id FROM elderly_card_trips WHERE ride_sttn_id IS NOT NULL
            UNION
            SELECT DISTINCT goff_sttn_id AS sttn_id FROM elderly_card_trips WHERE goff_sttn_id IS NOT NULL
        ),
        named AS (
            SELECT DISTINCT s.sttn_id, rs.sttn_nm, rs.ctpv_cd, rs.sgg_cd
            FROM stops s
            LEFT JOIN route_station rs ON rs.sttn_id = s.sttn_id
        )
        SELECT
            COUNT(*) AS card_trip_정류소수,
            SUM(CASE WHEN sc.lat IS NOT NULL THEN 1 ELSE 0 END) AS 좌표보유,
            ROUND(100.0 * SUM(CASE WHEN sc.lat IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct
        FROM named
        LEFT JOIN stop_coords sc 
          ON sc.ctpv_cd = named.ctpv_cd 
         AND sc.sgg_cd = named.sgg_cd 
         AND sc.sttn_nm = named.sttn_nm
    """).df())

=== 매칭 종류별 분포 ===
       matched_kind    cnt   pct
0  busStop_with_sgg  30689  88.4
1              none   2375   6.8
2  busStop_fallback   1659   4.8

=== ctpv_cd별 매칭률 ===
  ctpv_cd     전체       매칭   pct
0      11   8196   7635.0  93.2
1      28   3755   3547.0  94.5
2      41  22772  21166.0  92.9

=== 미매칭 정류소 샘플 (10개) ===
  ctpv_cd sgg_nm     sttn_nm matched_kind
0      41    안성시    반제리(미정차)         none
1      41    안성시          신촌         none
2      41    김포시   고촌IC(미정차)         none
3      41    김포시          동촌         none
4      41    안성시  하나로마트(미정차)         none
5      41    안성시   안성TG(미정차)         none
6      41    안성시    죽산리(미정차)         none
7      41    안성시          하동         none
8      41    김포시    구래리(미정차)         none
9      41    김포시   현수2교(미정차)         none

=== 좌표 범위 sanity check (수도권) ===
     lat_min    lat_max     lon_min     lon_max
0  35.062452  38.222304  126.219125  129.331005
  ※ 정상이면 lat 36.5~38.5, lon 126.5~127.7 범위

=== card_trip 정류소 매칭 적용 사전 검증 ===
   c

---

## ⚠️ 좌표 Outlier 재매칭 (버그 픽스)

기존 셀 2의 `find_stop_coords`는 시도 prefix 없이 `f"{sgg_nm} {sttn_nm}"`만 검색했음.
RGN_D2_NAME 필터만으로는 동명 sgg(예: 인천 동구 vs 부산 동구) 구별 불가 → 다른 시도로 새는 매칭 발생.

**해결**:
1. 좌표가 수도권 bbox(lat 36.5~38.5, lon 126.5~127.7) 밖인 행 추출
2. 검색어에 시도 prefix 추가: `f"{ctpv_short} {sgg_nm} {sttn_nm}"`
3. 필터에 `RGN_D1_NAME == ctpv_short` 추가
4. UPDATE

In [9]:
# ============================================================
# 셀 8 - Outlier 재매칭 (수도권 bbox 밖 좌표)
# ============================================================
# 셀 2의 find_stop_coords가 이제 시도 prefix + RGN_D1/D2 모두 검증하므로
# outlier만 추출해서 다시 호출하면 자동으로 정정됨.

# 수도권 bbox (느슨)
BBOX = {"lat_min": 36.4, "lat_max": 38.5, "lon_min": 125.9, "lon_max": 128.0}

with db_open(read_only=True) as con:
    outliers = con.execute(f"""
        SELECT ctpv_cd, sgg_cd, sgg_nm, sttn_nm, lat, lon, rgn_d1, rgn_d2, matched_name
        FROM stop_coords
        WHERE lat IS NOT NULL
          AND (lat < {BBOX['lat_min']} OR lat > {BBOX['lat_max']}
               OR lon < {BBOX['lon_min']} OR lon > {BBOX['lon_max']})
    """).df()

print(f"=== Outlier (수도권 bbox 밖): {len(outliers):,}개 ===")
if len(outliers) > 0:
    print("\n분포 (sgg_nm별):")
    print(outliers.groupby(['ctpv_cd', 'sgg_nm']).size().sort_values(ascending=False).head(20).to_string())
    print("\n샘플 5개 (잘못된 매칭):")
    print(outliers[['ctpv_cd','sgg_nm','sttn_nm','lat','lon','rgn_d1','rgn_d2']].head().to_string())

if len(outliers) > 0:
    print(f"\n=== Outlier {len(outliers):,}개 재매칭 시작 (find_stop_coords로) ===")
    BATCH = 30
    started = datetime.now()
    fixed_results = []
    for batch_start in range(0, len(outliers), BATCH):
        batch = outliers.iloc[batch_start:batch_start+BATCH]
        with ThreadPoolExecutor(max_workers=BATCH) as ex:
            futures = {
                ex.submit(find_stop_coords, r.ctpv_cd, r.sgg_nm, r.sttn_nm):
                (r.ctpv_cd, r.sgg_cd, r.sgg_nm, r.sttn_nm)
                for r in batch.itertuples(index=False)
            }
            for f in as_completed(futures):
                ctpv_cd, sgg_cd, sgg_nm, sttn_nm = futures[f]
                res = f.result()
                fixed_results.append({
                    "ctpv_cd": ctpv_cd, "sgg_cd": sgg_cd, "sgg_nm": sgg_nm, "sttn_nm": sttn_nm,
                    **res, "fetched_at": datetime.now(),
                })
        time.sleep(2 + random.random())
        if (batch_start // BATCH + 1) % 5 == 0:
            print(f"  진행: {min(batch_start+BATCH, len(outliers))}/{len(outliers)}")

    df = pd.DataFrame(fixed_results)
    with db_open() as con:
        con.register("df_view", df)
        con.execute("""
            DELETE FROM stop_coords
            WHERE (ctpv_cd, sgg_cd, sttn_nm) IN (
                SELECT ctpv_cd, sgg_cd, sttn_nm FROM df_view
            )
        """)
        con.execute("""
            INSERT INTO stop_coords
            SELECT ctpv_cd, sgg_cd, sgg_nm, sttn_nm,
                   matched_name, lat, lon, rgn_d1, rgn_d2,
                   matched_kind, query_used, fetched_at
            FROM df_view
        """)
        con.unregister("df_view")

    print(f"\n=== 재매칭 결과 ===")
    print(pd.Series([r["matched_kind"] for r in fixed_results]).value_counts().to_string())
    print(f"\n소요: {datetime.now() - started}")

    with db_open(read_only=True) as con:
        still = con.execute(f"""
            SELECT COUNT(*) FROM stop_coords WHERE lat IS NOT NULL
              AND (lat < {BBOX['lat_min']} OR lat > {BBOX['lat_max']}
                   OR lon < {BBOX['lon_min']} OR lon > {BBOX['lon_max']})
        """).fetchone()[0]
        print(f"\n재처리 후 여전히 bbox 밖: {still}개")
else:
    print("\nOutlier 없음 — 재매칭 불필요")

=== Outlier (수도권 bbox 밖): 0개 ===

Outlier 없음 — 재매칭 불필요


---

## 지하철역 좌표 수집

`route_station`은 버스 정류소만. 카드 trip의 지하철 sttn_id (예: `0150` = 서울역[1호선])는 `railway_station` 테이블에 있고 좌표 미수집.

- 대상: `railway_station` 수도권 (sarea_nm='수도권') unique sttn_nm
- 별도 테이블 `station_coords` (PK: ctpv_cd + sttn_nm — 시도 단위 분리)
- 검색 패턴: `f"{ctpv_short} {sttn_nm} {rte_nm}"` (예: "서울 시청역 2호선") + RGN_D1 검증
- **시도 prefix 필수**: "시청역"은 서울/부산 둘 다 있음 → 동명 disambiguation

In [12]:
# ============================================================
# 셀 10 - 지하철역 좌표 수집
# ============================================================
# railway_station sttn_nm 형식: "서울 [1호선]", "강남 [2호선]", "이촌(국립중앙박물관) [4호선]"
#   → [노선] 제거 + (보충) 제거 + "역" suffix 추가하여 검색
# ⚠️ 컬럼 순서 문제 회피 — DROP TABLE + 명시적 컬럼 리스트로 INSERT

import re as _re

def normalize_subway_name(raw: str) -> str:
    """railway_station sttn_nm을 카카오 검색용으로 변환.
    
    예시:
      "서울 [1호선]"               → "서울역"
      "이촌(국립중앙박물관) [4호선]"  → "이촌역"
      "테크노파크 [인천1호선]"        → "테크노파크역"
    """
    s = _re.sub(r'\s*\[[^\]]*\]\s*', '', raw).strip()  # [노선] 제거
    s = _re.sub(r'\s*\([^)]*\)\s*', '', s).strip()      # (보충) 제거
    if not s.endswith("역"):
        s += "역"
    return s


def find_subway_coords_v2(sttn_nm_clean: str, rte_nm: Optional[str] = None) -> dict:
    """수도권 지하철 매칭 — 정규화된 이름 기준."""
    SUDO_KWON = ["서울", "인천", "경기"]
    
    base_qs = []
    if rte_nm:
        for ctpv_p in SUDO_KWON:
            base_qs.append(f"{ctpv_p} {sttn_nm_clean} {rte_nm}")
    for ctpv_p in SUDO_KWON:
        base_qs.append(f"{ctpv_p} {sttn_nm_clean}")
    base_qs.append(sttn_nm_clean)
    
    for q in base_qs:
        j = kakao_search(q)
        if j is None:
            continue
        bs = j.get("busStop") or []
        m = next(
            (b for b in bs if any(b.get("RGN_D1_NAME", "").startswith(p) for p in SUDO_KWON)),
            None,
        )
        if m:
            return _pack(m, "subway_strict", q)
        pl = j.get("place") or []
        clean_no_yeok = sttn_nm_clean.removesuffix("역")
        m = next(
            (p for p in pl
             if (p.get("name", "").startswith(sttn_nm_clean) or p.get("name", "").startswith(clean_no_yeok))
             and any(prefix in (p.get("address") or "") for prefix in SUDO_KWON)),
            None,
        )
        if m:
            return {
                "matched_kind": "subway_place",
                "matched_name": m.get("name"),
                "lat": float(m["lat"]) if m.get("lat") else None,
                "lon": float(m["lon"]) if m.get("lon") else None,
                "rgn_d1": (m.get("address") or "").split()[0] if m.get("address") else None,
                "rgn_d2": None,
                "query_used": q,
            }
    
    return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
            "rgn_d1": None, "rgn_d2": None, "query_used": base_qs[0] if base_qs else ""}


# 테이블 새로 만들기 (DROP + CREATE — ALTER로 인한 컬럼 순서 문제 회피)
with db_open() as con:
    con.execute("DROP TABLE IF EXISTS station_coords")
    con.execute("""
    CREATE TABLE station_coords (
        ctpv_cd        TEXT,
        sttn_nm        TEXT,
        sttn_nm_clean  TEXT,
        rte_nm         TEXT,
        matched_name   TEXT,
        lat            DOUBLE,
        lon            DOUBLE,
        rgn_d1         TEXT,
        rgn_d2         TEXT,
        matched_kind   TEXT,
        query_used     TEXT,
        fetched_at     TIMESTAMP,
        PRIMARY KEY (ctpv_cd, sttn_nm)
    )
    """)
    print("station_coords 깨끗하게 재생성")

with db_open(read_only=True) as con:
    rail_targets = con.execute("""
        SELECT 
            '11' AS ctpv_cd,
            sttn_nm,
            ANY_VALUE(rte_nm) AS rte_nm
        FROM railway_station
        WHERE sarea_nm = '수도권' AND sttn_nm IS NOT NULL
        GROUP BY sttn_nm
        ORDER BY sttn_nm
    """).df()

rail_targets["sttn_nm_clean"] = rail_targets["sttn_nm"].apply(normalize_subway_name)
# 새로 만든 테이블이라 done은 항상 비어있음
rail_remaining = rail_targets.reset_index(drop=True)

print(f"\n수도권 지하철 unique 역: {len(rail_targets):,}개")
print(f"남은 작업: {len(rail_remaining):,}개")
print(f"\n샘플 (원본 → 정규화):")
print(rail_remaining[["sttn_nm", "sttn_nm_clean", "rte_nm"]].head(8).to_string())


n = len(rail_remaining)
if n == 0:
    print("\n남은 작업 없음")
else:
    print(f"\n=== 지하철역 {n}개 수집 시작 ===")
    BATCH = 20
    started = datetime.now()
    rail_results = []
    for batch_start in range(0, n, BATCH):
        batch = rail_remaining.iloc[batch_start:batch_start+BATCH]
        with ThreadPoolExecutor(max_workers=BATCH) as ex:
            futures = {
                ex.submit(find_subway_coords_v2, r.sttn_nm_clean, r.rte_nm):
                (r.ctpv_cd, r.sttn_nm, r.sttn_nm_clean, r.rte_nm)
                for r in batch.itertuples(index=False)
            }
            for f in as_completed(futures):
                ctpv_cd, sttn_nm, sttn_nm_clean, rte_nm = futures[f]
                res = f.result()
                rail_results.append({
                    "ctpv_cd": ctpv_cd, "sttn_nm": sttn_nm, "sttn_nm_clean": sttn_nm_clean,
                    "rte_nm": rte_nm, **res, "fetched_at": datetime.now(),
                })
        time.sleep(1.5 + random.random())
        if (batch_start // BATCH + 1) % 5 == 0:
            done_now = min(batch_start+BATCH, n)
            elapsed = datetime.now() - started
            stats_now = pd.Series([r['matched_kind'] for r in rail_results]).value_counts().to_dict()
            print(f"  [{done_now}/{n}] 경과 {str(elapsed).split('.')[0]} | {stats_now}")
    
    df = pd.DataFrame(rail_results)
    with db_open() as con:
        con.register("df_view", df)
        # 명시적 컬럼 리스트 — name-based 매칭으로 position 안 의존
        con.execute("""
            INSERT INTO station_coords (
                ctpv_cd, sttn_nm, sttn_nm_clean, rte_nm,
                matched_name, lat, lon, rgn_d1, rgn_d2,
                matched_kind, query_used, fetched_at
            )
            SELECT ctpv_cd, sttn_nm, sttn_nm_clean, rte_nm,
                   matched_name, lat, lon, rgn_d1, rgn_d2,
                   matched_kind, query_used, fetched_at
            FROM df_view
            ON CONFLICT (ctpv_cd, sttn_nm) DO NOTHING
        """)
        con.unregister("df_view")
    
    print(f"\n=== 지하철역 수집 완료 ===")
    print(pd.Series([r['matched_kind'] for r in rail_results]).value_counts().to_string())
    print(f"\n소요: {datetime.now() - started}")
    
    with db_open(read_only=True) as con:
        total = con.execute("SELECT COUNT(*) FROM station_coords").fetchone()[0]
    print(f"\n📊 station_coords 총 누적: {total:,}개")

station_coords 깨끗하게 재생성

수도권 지하철 unique 역: 774개
남은 작업: 774개

샘플 (원본 → 정규화):
              sttn_nm sttn_nm_clean   rte_nm
0  4.19민주묘지 [우이신설경전철]     4.19민주묘지역  우이신설경전철
1            가능 [1호선]           가능역      1호선
2          가락시장 [3호선]         가락시장역      3호선
3          가락시장 [8호선]         가락시장역      8호선
4       가산디지털단지 [1호선]      가산디지털단지역      1호선
5       가산디지털단지 [7호선]      가산디지털단지역      7호선
6            가양 [9호선]           가양역      9호선
7       가오리 [우이신설경전철]          가오리역  우이신설경전철

=== 지하철역 774개 수집 시작 ===
  [100/774] 경과 0:00:14 | {'subway_strict': 95, 'subway_place': 5}
  [200/774] 경과 0:00:32 | {'subway_strict': 189, 'subway_place': 8, 'none': 3}
  [300/774] 경과 0:00:52 | {'subway_strict': 281, 'subway_place': 15, 'none': 4}
  [400/774] 경과 0:01:10 | {'subway_strict': 377, 'subway_place': 19, 'none': 4}
  [500/774] 경과 0:01:29 | {'subway_strict': 471, 'subway_place': 25, 'none': 4}
  [600/774] 경과 0:01:46 | {'subway_strict': 568, 'subway_place': 28, 'none': 4}
  [700/774] 경과 0:02:03 | {'subway_

In [13]:
# ============================================================
# 셀 11 - 경춘선 등 강원 구간 지하철역 보충 매칭
# ============================================================
# 셀 10이 SUDO_KWON=[서울,인천,경기]만 시도해서 강원 구간(경춘선 굴봉산/백양리/김유정/남춘천) 못 잡음.
# 'none' 마킹된 것만 추출 → 강원 prefix 포함해서 재시도 → UPDATE

def find_subway_coords_relaxed(sttn_nm_clean: str, rte_nm: Optional[str] = None) -> dict:
    """수도권 + 인접(강원 등) 시도 포함 지하철 매칭."""
    EXTENDED = ["서울", "인천", "경기", "강원", "충남", "충북"]
    
    base_qs = []
    if rte_nm:
        for ctpv_p in EXTENDED:
            base_qs.append(f"{ctpv_p} {sttn_nm_clean} {rte_nm}")
    for ctpv_p in EXTENDED:
        base_qs.append(f"{ctpv_p} {sttn_nm_clean}")
    base_qs.append(sttn_nm_clean)
    
    for q in base_qs:
        j = kakao_search(q)
        if j is None:
            continue
        bs = j.get("busStop") or []
        m = next(
            (b for b in bs if any(b.get("RGN_D1_NAME", "").startswith(p) for p in EXTENDED)),
            None,
        )
        if m:
            return _pack(m, "subway_strict", q)
        pl = j.get("place") or []
        clean_no_yeok = sttn_nm_clean.removesuffix("역")
        m = next(
            (p for p in pl
             if (p.get("name", "").startswith(sttn_nm_clean) or p.get("name", "").startswith(clean_no_yeok))
             and any(prefix in (p.get("address") or "") for prefix in EXTENDED)),
            None,
        )
        if m:
            return {
                "matched_kind": "subway_place",
                "matched_name": m.get("name"),
                "lat": float(m["lat"]) if m.get("lat") else None,
                "lon": float(m["lon"]) if m.get("lon") else None,
                "rgn_d1": (m.get("address") or "").split()[0] if m.get("address") else None,
                "rgn_d2": None,
                "query_used": q,
            }
    
    return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
            "rgn_d1": None, "rgn_d2": None, "query_used": base_qs[0] if base_qs else ""}


# 'none' 행 추출
with db_open(read_only=True) as con:
    none_subway = con.execute("""
        SELECT ctpv_cd, sttn_nm, sttn_nm_clean, rte_nm
        FROM station_coords
        WHERE matched_kind = 'none'
    """).df()

print(f"=== 미매칭 지하철역: {len(none_subway)}개 ===")
print(none_subway.to_string())

if len(none_subway) > 0:
    print(f"\n재시도 (강원/충청 prefix 포함)...")
    fixed = []
    for r in none_subway.itertuples(index=False):
        res = find_subway_coords_relaxed(r.sttn_nm_clean, r.rte_nm)
        fixed.append({
            "ctpv_cd": r.ctpv_cd, "sttn_nm": r.sttn_nm, "sttn_nm_clean": r.sttn_nm_clean,
            "rte_nm": r.rte_nm, **res, "fetched_at": datetime.now(),
        })
        time.sleep(0.5 + random.random() * 0.5)

    df_fix = pd.DataFrame(fixed)
    print("\n결과:")
    print(df_fix[["sttn_nm", "matched_kind", "matched_name", "lat", "lon", "rgn_d1"]].to_string())

    # UPDATE: DELETE 기존 + INSERT
    with db_open() as con:
        con.register("df_view", df_fix)
        con.execute("""
            DELETE FROM station_coords
            WHERE (ctpv_cd, sttn_nm) IN (
                SELECT ctpv_cd, sttn_nm FROM df_view
            )
        """)
        con.execute("""
            INSERT INTO station_coords (
                ctpv_cd, sttn_nm, sttn_nm_clean, rte_nm,
                matched_name, lat, lon, rgn_d1, rgn_d2,
                matched_kind, query_used, fetched_at
            )
            SELECT ctpv_cd, sttn_nm, sttn_nm_clean, rte_nm,
                   matched_name, lat, lon, rgn_d1, rgn_d2,
                   matched_kind, query_used, fetched_at
            FROM df_view
        """)
        con.unregister("df_view")
    print("\n✅ 업데이트 완료")

    # 최종 확인
    with db_open(read_only=True) as con:
        print("\n=== station_coords 최종 분포 ===")
        print(con.execute("""
            SELECT matched_kind, COUNT(*) AS cnt
            FROM station_coords GROUP BY matched_kind ORDER BY cnt DESC
        """).df().to_string())
else:
    print("\nnone 없음 — 보충 불필요")

=== 미매칭 지하철역: 4개 ===
  ctpv_cd    sttn_nm sttn_nm_clean rte_nm
0      11  굴봉산 [경춘선]          굴봉산역    경춘선
1      11  백양리 [경춘선]          백양리역    경춘선
2      11  김유정 [경춘선]          김유정역    경춘선
3      11  남춘천 [경춘선]          남춘천역    경춘선

재시도 (강원/충청 prefix 포함)...

결과:
     sttn_nm   matched_kind matched_name        lat         lon   rgn_d1
0  굴봉산 [경춘선]  subway_strict         굴봉산역  37.832076  127.557587  강원특별자치도
1  백양리 [경춘선]  subway_strict         백양리역  37.831179  127.588885  강원특별자치도
2  김유정 [경춘선]  subway_strict         김유정역  37.817431  127.715033  강원특별자치도
3  남춘천 [경춘선]  subway_strict     남춘천역2번출구  37.863864  127.724711  강원특별자치도

✅ 업데이트 완료

=== station_coords 최종 분포 ===
    matched_kind  cnt
0  subway_strict  740
1   subway_place   34


---

## 버스 미매칭 보충 (sgg 잘못 + 정류소명 변형)

기존 `find_stop_coords`는 **sgg 정확 일치**를 요구해서 누락된 케이스 많음:
1. **DB sgg_cd 자체가 잘못 등록** — 예: `관악구보훈회관.신림푸르지오`가 DB는 금천구(11545), 카카오는 관악구. 시도만 일치해도 채택해야 살릴 수 있음
2. **정류소명에 `(미정차)`, `(가상)` 표시** — 떼면 매칭 가능한 경우
3. **양방향 suffix `B`/`A`** — 떼면 매칭 가능
4. **점(.)으로 분리** — 첫 토큰 또는 두번째 토큰만으로 매칭

### 처리 단계 (각 정류소마다 순서대로 시도)
1. 정규화 (괄호제거, B/A 제거) + 시도 일치 검색 → `busStop_loose`
2. 점 분리 첫 토큰 + 시도 일치 → `busStop_token1`
3. 점 분리 두번째 토큰 + 시도 일치 → `busStop_token2`
4. 그래도 안 잡히면 'none' 유지

`(미정차)`/`(가상)` 같이 카카오에 진짜 없는 건 어차피 미매칭. 영향 큰 정류소 220개 중 70-80% 살릴 듯.

In [14]:
# ============================================================
# 셀 13 - 버스 미매칭 정류소 보충 매칭
# ============================================================

def normalize_stop_name(raw: str) -> str:
    """버스 정류소명 정규화: (미정차)/(가상) 등 괄호 제거, 양방향 B/A suffix 제거."""
    s = re.sub(r'\s*\([^)]*\)\s*', ' ', raw).strip()  # (...) 제거
    s = re.sub(r'\s*\[[^\]]*\]\s*', ' ', s).strip()    # [...] 제거
    s = re.sub(r'[ABab]$', '', s).strip()              # 끝의 B/A 제거
    return s


def split_dot(name: str) -> list:
    """'A.B.C' → ['A','B','C']. 점 없으면 빈 리스트."""
    if '.' not in name:
        return []
    return [t.strip() for t in name.split('.') if t.strip()]


def find_stop_coords_loose(ctpv_cd: str, sgg_nm: str, sttn_nm: str) -> dict:
    """sgg 정확 일치 안 해도 시도 일치만으로 채택. 정규화 + 토큰 분리 fallback 포함."""
    ctpv_short = CTPV_SHORT.get(ctpv_cd)
    if ctpv_short is None:
        return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
                "rgn_d1": None, "rgn_d2": None, "query_used": ""}

    SUDO_KWON = ["서울", "인천", "경기"]
    norm_name = normalize_stop_name(sttn_nm)
    
    # 시도해볼 검색어 목록 (우선순위 순)
    queries = []
    # 1단계: 정규화된 이름
    if norm_name and norm_name != sttn_nm:
        queries.append(("loose_normalized", f"{ctpv_short} {sgg_nm} {norm_name}"))
        queries.append(("loose_normalized", f"{ctpv_short} {norm_name}"))
    # 2단계: 점 분리 토큰
    tokens = split_dot(norm_name or sttn_nm)
    for i, tok in enumerate(tokens):
        kind = f"token{i+1}"
        if len(tok) >= 2:  # 너무 짧으면 패스 (모호)
            queries.append((kind, f"{ctpv_short} {sgg_nm} {tok}"))
            queries.append((kind, f"{ctpv_short} {tok}"))

    for kind, q in queries:
        j = kakao_search(q)
        if j is None:
            continue
        bs = j.get("busStop") or []
        # 시도(서울/인천/경기) 일치만 검증 — sgg는 안 봐도 됨 (느슨)
        m = next(
            (b for b in bs if any(b.get("RGN_D1_NAME", "").startswith(p) for p in SUDO_KWON)),
            None,
        )
        if m:
            return _pack(m, f"busStop_{kind}", q)

    return {"matched_kind": "none", "matched_name": None, "lat": None, "lon": None,
            "rgn_d1": None, "rgn_d2": None, "query_used": queries[0][1] if queries else ""}


# 'none'인 버스 정류소 추출
with db_open(read_only=True) as con:
    none_bus = con.execute("""
        SELECT ctpv_cd, sgg_cd, sgg_nm, sttn_nm
        FROM stop_coords
        WHERE matched_kind = 'none'
    """).df()

print(f"=== 버스 미매칭: {len(none_bus):,}개 ===")
print(f"\n정규화 샘플:")
sample = none_bus.head(8).copy()
sample['normalized'] = sample['sttn_nm'].apply(normalize_stop_name)
sample['tokens'] = sample['sttn_nm'].apply(lambda s: split_dot(normalize_stop_name(s)))
print(sample[['sgg_nm', 'sttn_nm', 'normalized', 'tokens']].to_string())


if len(none_bus) > 0:
    print(f"\n=== 보충 매칭 시작 ({len(none_bus):,}개) ===")
    BATCH = 30
    started = datetime.now()
    fixed = []
    for batch_start in range(0, len(none_bus), BATCH):
        batch = none_bus.iloc[batch_start:batch_start+BATCH]
        with ThreadPoolExecutor(max_workers=BATCH) as ex:
            futures = {
                ex.submit(find_stop_coords_loose, r.ctpv_cd, r.sgg_nm, r.sttn_nm):
                (r.ctpv_cd, r.sgg_cd, r.sgg_nm, r.sttn_nm)
                for r in batch.itertuples(index=False)
            }
            for f in as_completed(futures):
                ctpv_cd, sgg_cd, sgg_nm, sttn_nm = futures[f]
                res = f.result()
                fixed.append({
                    "ctpv_cd": ctpv_cd, "sgg_cd": sgg_cd, "sgg_nm": sgg_nm, "sttn_nm": sttn_nm,
                    **res, "fetched_at": datetime.now(),
                })
        time.sleep(2 + random.random())
        if (batch_start // BATCH + 1) % 5 == 0:
            done_now = min(batch_start+BATCH, len(none_bus))
            elapsed = datetime.now() - started
            stats_now = pd.Series([r['matched_kind'] for r in fixed]).value_counts().to_dict()
            print(f"  [{done_now}/{len(none_bus)}] 경과 {str(elapsed).split('.')[0]} | {stats_now}")

    df_fix = pd.DataFrame(fixed)
    
    # UPDATE
    with db_open() as con:
        con.register("df_view", df_fix)
        con.execute("""
            DELETE FROM stop_coords
            WHERE (ctpv_cd, sgg_cd, sttn_nm) IN (
                SELECT ctpv_cd, sgg_cd, sttn_nm FROM df_view
            )
        """)
        con.execute("""
            INSERT INTO stop_coords (
                ctpv_cd, sgg_cd, sgg_nm, sttn_nm,
                matched_name, lat, lon, rgn_d1, rgn_d2,
                matched_kind, query_used, fetched_at
            )
            SELECT ctpv_cd, sgg_cd, sgg_nm, sttn_nm,
                   matched_name, lat, lon, rgn_d1, rgn_d2,
                   matched_kind, query_used, fetched_at
            FROM df_view
        """)
        con.unregister("df_view")
    
    print(f"\n=== 보충 결과 ===")
    print(pd.Series([r["matched_kind"] for r in fixed]).value_counts().to_string())
    print(f"\n소요: {datetime.now() - started}")

    # 최종 분포
    with db_open(read_only=True) as con:
        print("\n=== stop_coords 최종 분포 ===")
        print(con.execute("""
            SELECT matched_kind, COUNT(*) AS cnt,
                   ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
            FROM stop_coords GROUP BY matched_kind ORDER BY cnt DESC
        """).df().to_string())
        
        # outlier 재확인
        BBOX = {"lat_min": 36.4, "lat_max": 38.5, "lon_min": 125.9, "lon_max": 128.0}
        outlier = con.execute(f"""
            SELECT COUNT(*) FROM stop_coords WHERE lat IS NOT NULL
              AND (lat < {BBOX['lat_min']} OR lat > {BBOX['lat_max']}
                   OR lon < {BBOX['lon_min']} OR lon > {BBOX['lon_max']})
        """).fetchone()[0]
        print(f"\n수도권 bbox 밖 outlier: {outlier}개")
else:
    print("\nnone 없음 — 보충 불필요")

=== 버스 미매칭: 2,379개 ===

정규화 샘플:
  sgg_nm     sttn_nm normalized tokens
0    안성시    반제리(미정차)        반제리     []
1    안성시          신촌         신촌     []
2    김포시   고촌IC(미정차)       고촌IC     []
3    김포시          동촌         동촌     []
4    안성시  하나로마트(미정차)      하나로마트     []
5    안성시   안성TG(미정차)       안성TG     []
6    안성시    죽산리(미정차)        죽산리     []
7    안성시          하동         하동     []

=== 보충 매칭 시작 (2,379개) ===
  [150/2379] 경과 0:00:18 | {'none': 102, 'busStop_loose_normalized': 43, 'busStop_token1': 4, 'busStop_token2': 1}
  [300/2379] 경과 0:00:36 | {'none': 202, 'busStop_loose_normalized': 84, 'busStop_token1': 11, 'busStop_token2': 3}
  [450/2379] 경과 0:00:53 | {'none': 310, 'busStop_loose_normalized': 117, 'busStop_token1': 17, 'busStop_token2': 6}
  [600/2379] 경과 0:01:12 | {'none': 416, 'busStop_loose_normalized': 149, 'busStop_token1': 27, 'busStop_token2': 8}
  [750/2379] 경과 0:01:30 | {'none': 506, 'busStop_loose_normalized': 200, 'busStop_token1': 35, 'busStop_token2': 9}
  [900/2379] 